# NB4 — Medallion Pipeline (Bronze → Silver → Gold), lightweight

**Use case:** LLM observability — exact schema from slide §8 (Lakehouse cho AI/ML) medallion frame.
Maps to deliverable bullet 4 (the Milestone-1 Lakehouse artifact).

Pre-req: ran `make data` — but if you jumped straight here, the cell below
generates the Bronze sample for you rather than failing on a missing path.

In [1]:
import _setup  # noqa: F401  -- adds scripts/ to sys.path
from pathlib import Path

import polars as pl
import duckdb
from deltalake import DeltaTable, write_deltalake
from lakehouse import path, reset

BRONZE = path("bronze", "llm_calls_raw")
SILVER = path("silver", "llm_calls")
GOLD   = path("gold",   "llm_daily_metrics")

# Self-healing pre-req (same pattern as NB7/NB8). Without this, skipping
# `make data` surfaces as a raw `Os { code: 2, kind: NotFound }` from the Rust
# layer — technically correct, useless to a student.
if not Path(BRONZE).exists():
    print("Bronze not found — running scripts/generate_data_lite.py first ...")
    import generate_data_lite

    generate_data_lite.main()

## Bronze — verify raw is loaded

In [2]:
bronze_n = DeltaTable(BRONZE).to_pyarrow_table().num_rows
print(f"Bronze rows: {bronze_n:,}")
print(pl.from_arrow(DeltaTable(BRONZE).to_pyarrow_table().slice(0, 2)))

Bronze rows: 200,000
shape: (2, 3)
┌─────────────────────────────────┬─────────────────────────┬─────────────────────────────────┐
│ request_id                      ┆ ts                      ┆ raw_json                        │
│ ---                             ┆ ---                     ┆ ---                             │
│ str                             ┆ datetime[μs, UTC]       ┆ str                             │
╞═════════════════════════════════╪═════════════════════════╪═════════════════════════════════╡
│ 2c4a0ab5-79ae-45f0-a4b6-07ac67… ┆ 2026-04-01 00:00:00 UTC ┆ {"model": "claude-sonnet-4-6",… │
│ b7cce488-a49b-4d8e-8b41-b05810… ┆ 2026-04-01 00:00:03 UTC ┆ {"model": "claude-haiku-4-5", … │
└─────────────────────────────────┴─────────────────────────┴─────────────────────────────────┘


## Silver — parse, validate, dedup

Rules: drop malformed JSON, dedupe by `request_id`, project typed columns.

In [3]:
reset(SILVER)

# DuckDB does the JSON parse + dedup in one query — Polars also works,
# DuckDB just has nicer JSON syntax for this case.
# DuckDB reads Delta through Arrow, not through `delta_scan()`. The latter
# autoloads an extension over the network; Arrow registration is offline and
# zero-copy, so the lab works on a locked-down machine.
con = duckdb.connect()
# `ts` is TIMESTAMPTZ and CAST(ts AS DATE) uses the session time zone. Without
# this, a laptop in Asia/Ho_Chi_Minh buckets 7 UTC days into 8 partial local
# days (04-01 with 17 h, 04-08 with 7 h). Pin the day boundary to UTC.
con.sql("SET TimeZone = 'UTC'")
con.register("bronze", DeltaTable(BRONZE).to_pyarrow_table())

silver_arrow = con.sql(f"""
    WITH parsed AS (
      SELECT
        request_id,
        ts,
        CAST(ts AS DATE)                            AS date,
        json_extract_string(raw_json, '$.model')          AS model,
        json_extract_string(raw_json, '$.user_id')        AS user_id,
        CAST(json_extract(raw_json, '$.usage.input')  AS INTEGER) AS prompt_tokens,
        CAST(json_extract(raw_json, '$.usage.output') AS INTEGER) AS completion_tokens,
        CAST(json_extract(raw_json, '$.latency_ms')   AS INTEGER) AS latency_ms,
        json_extract_string(raw_json, '$.status')         AS status,
        ROW_NUMBER() OVER (PARTITION BY request_id ORDER BY ts) AS rn
      FROM bronze
    )
    SELECT request_id, ts, date, model, user_id,
           prompt_tokens, completion_tokens, latency_ms, status
    FROM parsed
    WHERE rn = 1 AND model IS NOT NULL
""").arrow()

write_deltalake(SILVER, silver_arrow, mode="overwrite", partition_by=["date"])

silver_n = DeltaTable(SILVER).to_pyarrow_table().num_rows
print(f"Silver rows: {silver_n:,}  (Bronze {bronze_n:,} → dedup dropped {bronze_n - silver_n:,})")
assert silver_n < bronze_n, (
    "Silver has the same row count as Bronze — dedup did not run. "
    "Did you regenerate Bronze with the latest generator (which injects retries)?"
)

Silver rows: 190,052  (Bronze 200,000 → dedup dropped 9,948)


## Gold — aggregate to (date, model) metrics

In [4]:
reset(GOLD)

# Illustrative cost model — NOT canonical pricing.
# (input USD / 1M tokens, output USD / 1M tokens)
COST_TABLE = """
  VALUES
    ('claude-haiku-4-5',  0.80,  4.00),
    ('claude-sonnet-4-6', 3.00, 15.00),
    ('claude-opus-4-7', 15.00, 75.00)
"""

con.register("silver", DeltaTable(SILVER).to_pyarrow_table())
gold_arrow = con.sql(f"""
    WITH cost(model, c_in, c_out) AS ({COST_TABLE})
    SELECT
      s.date,
      s.model,
      QUANTILE_CONT(s.latency_ms, 0.50) AS p50_latency_ms,
      QUANTILE_CONT(s.latency_ms, 0.95) AS p95_latency_ms,
      SUM(s.prompt_tokens)              AS total_prompt_tokens,
      SUM(s.completion_tokens)          AS total_completion_tokens,
      AVG(CASE WHEN s.status <> 'ok' THEN 1.0 ELSE 0.0 END) AS error_rate,
      (SUM(s.prompt_tokens)     * c.c_in  / 1e6) +
      (SUM(s.completion_tokens) * c.c_out / 1e6) AS cost_usd
    FROM silver s
    JOIN cost c USING (model)
    GROUP BY s.date, s.model, c.c_in, c.c_out
    ORDER BY s.date, s.model
""").arrow()

write_deltalake(GOLD, gold_arrow, mode="overwrite", partition_by=["date"])

# Z-order for fast filter-by-model dashboards
DeltaTable(GOLD).optimize.z_order(["model"])

{'numFilesAdded': 7,
 'numFilesRemoved': 7,
 'filesAdded': '{"avg":2769.0,"max":2770,"min":2766,"totalFiles":7,"totalSize":19383}',
 'filesRemoved': '{"avg":2688.1428571428573,"max":2689,"min":2687,"totalFiles":7,"totalSize":18817}',
 'partitionsOptimized': 7,
 'numBatches': 7,
 'totalConsideredFiles': 7,
 'totalFilesSkipped': 0,
 'preserveInsertionOrder': False,
 'plannerStrategy': 'zOrder',
 'preservedStableOrder': False,
 'maxBinSpanFiles': 1}

## Verify Gold

In [5]:
gold_df = pl.from_arrow(DeltaTable(GOLD).to_pyarrow_table())
print(gold_df)

# Slide-5 deliverable: "Gold p50/p95/cost qua ≥ 7 ngày". Make that explicit.
n_dates = gold_df.select("date").n_unique()
n_models = gold_df.select("model").n_unique()
print(
    f"\n──── Gold deliverable metrics ────\n"
    f"  Distinct dates:   {n_dates:>3}   (target ≥ 7)\n"
    f"  Distinct models:  {n_models:>3}\n"
    f"  Total Gold rows:  {gold_df.height:>3}   (= dates × models)"
)
assert n_dates >= 7, (
    f"Gold has only {n_dates} dates — slide deliverable requires ≥ 7. "
    "Re-run `make data` (the generator spreads across 7 UTC days)."
)

shape: (21, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ date       ┆ model      ┆ p50_latenc ┆ p95_laten ┆ total_pro ┆ total_com ┆ error_rat ┆ cost_usd  │
│ ---        ┆ ---        ┆ y_ms       ┆ cy_ms     ┆ mpt_token ┆ pletion_t ┆ e         ┆ ---       │
│ date       ┆ str        ┆ ---        ┆ ---       ┆ s         ┆ okens     ┆ ---       ┆ f64       │
│            ┆            ┆ f64        ┆ f64       ┆ ---       ┆ ---       ┆ f64       ┆           │
│            ┆            ┆            ┆           ┆ decimal[3 ┆ decimal[3 ┆           ┆           │
│            ┆            ┆            ┆           ┆ 8,0]      ┆ 8,0]      ┆           ┆           │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 2026-04-06 ┆ claude-hai ┆ 562.0      ┆ 1135.45   ┆ 16374102  ┆ 8149922   ┆ 0.045499  ┆ 45.69897  │
│            ┆ ku-4-5     ┆            ┆           ┆           ┆           ┆

## ✅ Deliverable check
- [ ] All three tables exist under `_lakehouse/{bronze,silver,gold}/`
- [ ] Silver has fewer rows than Bronze (dedup worked)
- [ ] Gold spans ≥ 7 dates × 3 models (slide §8 medallion contract)
- [ ] Cost & error_rate columns populated and non-zero

## Full Gold contract check

The cells above only assert ≥ 7 dates. This cell checks every Gold requirement
explicitly: three tables on storage, Silver < Bronze, full date × model grid,
p50 ≤ p95, cost_usd > 0 and error_rate within [0, 1].

In [6]:
from pathlib import Path as _P

with pl.Config(tbl_rows=-1, tbl_width_chars=140, fmt_str_lengths=20):
    print(gold_df.sort("date", "model").select(
        "date", "model", "p50_latency_ms", "p95_latency_ms", "error_rate",
        pl.col("cost_usd").round(2)))

_tables = {name: (_P(p) / "_delta_log").is_dir() for name, p in
           [("bronze", BRONZE), ("silver", SILVER), ("gold", GOLD)]}
gold_checks = {
    "bronze/silver/gold all on storage": all(_tables.values()),
    "silver < bronze (dedup)": silver_n < bronze_n,
    "≥ 7 dates × 3 models, full grid": n_dates >= 7 and n_models == 3
                                       and gold_df.height == n_dates * n_models,
    "p50 ≤ p95 on every row": gold_df.filter(pl.col("p50_latency_ms") > pl.col("p95_latency_ms")).height == 0,
    "cost_usd > 0 on every row": gold_df.filter(pl.col("cost_usd") <= 0).height == 0,
    "error_rate in [0, 1] on every row": gold_df.filter(~pl.col("error_rate").is_between(0, 1)).height == 0,
}
for k, v in gold_checks.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")
for name, p in [("bronze", BRONZE), ("silver", SILVER), ("gold", GOLD)]:
    print(f"  {name:6s} → {p}")
assert all(gold_checks.values()), "NB4 Gold contract incomplete — see FAIL rows above"
print("\nNB4 complete.")

shape: (21, 6)
┌────────────┬───────────────────┬────────────────┬────────────────┬────────────┬──────────┐
│ date       ┆ model             ┆ p50_latency_ms ┆ p95_latency_ms ┆ error_rate ┆ cost_usd │
│ ---        ┆ ---               ┆ ---            ┆ ---            ┆ ---        ┆ ---      │
│ date       ┆ str               ┆ f64            ┆ f64            ┆ f64        ┆ f64      │
╞════════════╪═══════════════════╪════════════════╪════════════════╪════════════╪══════════╡
│ 2026-04-01 ┆ claude-haiku-4-5  ┆ 563.0          ┆ 1131.0         ┆ 0.048825   ┆ 46.41    │
│ 2026-04-01 ┆ claude-opus-4-7   ┆ 3008.0         ┆ 6039.4         ┆ 0.05013    ┆ 284.78   │
│ 2026-04-01 ┆ claude-sonnet-4-6 ┆ 1381.0         ┆ 2751.65        ┆ 0.050467   ┆ 345.49   │
│ 2026-04-02 ┆ claude-haiku-4-5  ┆ 578.0          ┆ 1142.05        ┆ 0.049259   ┆ 46.25    │
│ 2026-04-02 ┆ claude-opus-4-7   ┆ 3013.0         ┆ 5950.8         ┆ 0.052444   ┆ 295.02   │
│ 2026-04-02 ┆ claude-sonnet-4-6 ┆ 1384.0         ┆ 273

## Kết quả & giải thích (Đỗ Quang Vinh)

| Tầng | Vị trí | Số dòng |
|---|---|---|
| Bronze | `_lakehouse/bronze/llm_calls_raw` | **200 000** (JSON thô) |
| Silver | `_lakehouse/silver/llm_calls` (partition theo `date`) | **190 052** (dedup bỏ 9 948) |
| Gold | `_lakehouse/gold/llm_daily_metrics` | **21 = 7 ngày × 3 model** |

Cell "Full Gold contract check" xác nhận cả 6 điều kiện: đủ 3 bảng trên storage, Silver < Bronze, lưới đủ 7 × 3, p50 ≤ p95 ở mọi dòng (vd. opus ~3 000 / ~6 000 ms, haiku ~560 / ~1 130 ms), `cost_usd > 0` (haiku ~$46/ngày, sonnet ~$345, opus ~$285) và `error_rate` ∈ [0, 1] (≈ 0.045–0.062).

**Dedup ở Silver giải quyết gì?** Generator chèn 9 948 bản ghi trùng `request_id` (mô phỏng client retry / at-least-once delivery). Nếu không dedup, token, chi phí và số request bị đếm hai lần. `ROW_NUMBER() OVER (PARTITION BY request_id ORDER BY ts) = 1` giữ lần ghi nhận đầu tiên. Bronze vẫn giữ bản trùng để có thể audit và chạy lại.

**Vì sao dashboard đọc Gold?** Gold đã tổng hợp sẵn theo (ngày, model): 21 dòng thay vì quét 190K dòng và parse JSON mỗi lần mở dashboard. Định nghĩa metric (p95, cách tính error rate, bảng giá) nằm ở một chỗ, nên mọi dashboard cho cùng một con số.

**Cách tính có phù hợp dữ liệu đầu vào không? — mình tìm ra một lỗi và đã sửa.**
1. *Ranh giới ngày:* `ts` là TIMESTAMPTZ, còn `CAST(ts AS DATE)` dùng múi giờ của phiên DuckDB. Trên máy mình (Asia/Saigon) bản gốc cho **8 ngày × 3 = 24 dòng**: ngày 04-01 chỉ có 17 giờ (haiku $33 thay vì ~$46) và ngày 04-08 chỉ có 7 giờ ($13). Dữ liệu thật ra là 7 ngày UTC tròn. Mình thêm `SET TimeZone = 'UTC'` nên giờ có đúng 7 ngày đủ 24 giờ. Nếu không sửa, kết quả Gold phụ thuộc máy chạy, và hai ngày biên trông như chi phí "giảm" dù thực ra là do thiếu giờ.
2. *Error rate:* `status <> 'ok'` gộp `error` (3 757) với `rate_limited` (5 683), tức hơn 60% "lỗi" thực chất là bị giới hạn tốc độ. Phù hợp nếu câu hỏi là "request không thành công", nhưng nên tách hai cột để phân biệt lỗi model với thiếu quota. Ngoài ra `status` NULL sẽ bị tính là thành công.
3. *Chi phí:* dùng bảng giá minh hoạ của lab (không phải giá thật), tính trên token của request đã dedup. Request bị lỗi vẫn bị tính token, đúng với cách nhà cung cấp tính tiền input.